# Milestone 2: Data Quality dan Profiling

Lanjutkan setelah notebook 01. Polars menangani transformasi; DuckDB memprofilkan
data sebelum/sesudah cleaning dan memvalidasi aturan kualitas. CSV asli tidak ditimpa.


In [1]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (current, *current.parents)
     if (path / 'requirements.txt').is_file() and (path / 'data/README.md').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Buka Jupyter dari akar repository atau folder notebooks/.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import polars as pl
from IPython.display import Markdown, display
from src.common import (
    connect_analytics, prepare_directories, read_json, sha256_file,
    sink_parquet_atomic, sql_literal, write_json,
)
prepare_directories(PROJECT_ROOT)

from src.dataset import ANALYSIS_COLUMNS, BUSINESS_KEY, missing_summary, normalise_news

DATA_PATH = PROJECT_ROOT / 'data/processed/dataset_raw.parquet'
NORMALISED_PATH = PROJECT_ROOT / 'data/processed/dataset_normalised.parquet'
CANDIDATE_PATH = PROJECT_ROOT / 'data/processed/dataset_candidate.parquet'
OUTPUT_PATH = PROJECT_ROOT / 'data/processed/dataset_bersih.parquet'
PROFILE = read_json(PROJECT_ROOT / 'output/profiling_summary.json')

if not DATA_PATH.is_file():
    raise FileNotFoundError('Jalankan notebook 01 untuk menghasilkan dataset_raw.parquet.')
assert PROFILE['size_requirement_met'], 'Input pada milestone 1 belum memenuhi syarat skala.'
assert sha256_file(DATA_PATH) == PROFILE['raw_parquet_sha256'], 'Parquet berubah; ulangi notebook 01.'

df = pl.scan_parquet(DATA_PATH)
raw_rows = int(df.select(pl.len()).collect(engine='streaming').item())
assert raw_rows == PROFILE['rows_raw']
print(f'Ukuran awal: {raw_rows:,} artikel')


Ukuran awal: 455,678 artikel


## 1. Profiling dengan DuckDB `SUMMARIZE`

DuckDB membaca Parquet langsung; teks penuh tidak dimuat menjadi DataFrame Python.


In [2]:
connection = connect_analytics(PROJECT_ROOT)
connection.execute(
    f'CREATE VIEW dataset AS SELECT * FROM read_parquet({sql_literal(DATA_PATH)})'
)
profil = connection.execute('SUMMARIZE SELECT * FROM dataset').pl()
profil.write_csv(PROJECT_ROOT / 'output/profile_before_cleaning.csv')
display(profil)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""row_id""","""UINTEGER""","""0""","""455677""",475106,"""227838.5""","""131543.05231938348""","""114240""","""225436""","""340747""",455678,0.00
"""published_raw""","""VARCHAR""","""2017-01-01 00:00:00""","""2017-12-31 23:58:25""",401317,null,null,null,null,null,455678,0.00
"""title_raw""","""VARCHAR""",""""" Di Sini Punya Prinsip, Jika …","""﻿﻿Waspada akan Diabetes Mellit…",526630,null,null,null,null,null,455678,0.00
"""content_raw""","""VARCHAR""","""&amp;ldquo;Siapa di antara kit…",""" """,419110,null,null,null,null,null,455678,0.00
"""source_raw""","""VARCHAR""","""beritasatu.com""","""tribunnews.com""",6,null,null,null,null,null,455678,0.00
"""url_raw""","""VARCHAR""","""http://edukasi.kompas.com/read…","""https://www.tempo.co/read/1046…",390541,null,null,null,null,null,455678,0.00


## 2. Deteksi Missing Values

Normalisasi spasi/HTML, parse tanggal dengan format eksplisit, turunkan jumlah kata,
dan identifikasi sumber. Nilai raw tetap tersimpan dalam tahap normalisasi untuk audit.


In [3]:
sink_parquet_atomic(normalise_news(df), NORMALISED_PATH)
normalised = pl.scan_parquet(NORMALISED_PATH)
null_summary = missing_summary(normalised, ['published_at', 'title', 'content', 'source_text'])
null_summary.write_csv(PROJECT_ROOT / 'output/nulls_normalised.csv')
display(null_summary)
display(normalised.select('published_raw', 'published_at', 'has_time').head(10)
        .collect(engine='streaming'))


kolom,jumlah_null,persen_null
str,i64,f64
"""content""",52,0.0114
"""published_at""",0,0.0
"""title""",0,0.0
"""source_text""",0,0.0


published_raw,published_at,has_time
str,datetime[μs],bool
"""2017-01-01 00:00:00""",2017-01-01 00:00:00,true
"""2017-01-01 00:01:00""",2017-01-01 00:01:00,true
"""2017-01-01 00:03:09""",2017-01-01 00:03:09,true
"""2017-01-01 00:04:21""",2017-01-01 00:04:21,true
"""2017-01-01 00:05:08""",2017-01-01 00:05:08,true
"""2017-01-01 00:05:54""",2017-01-01 00:05:54,true
"""2017-01-01 00:16:52""",2017-01-01 00:16:52,true
"""2017-01-01 00:16:54""",2017-01-01 00:16:54,true
"""2017-01-01 00:20:52""",2017-01-01 00:20:52,true


## 3. Penanganan Missing Values

Tanggal gagal parse, tanggal di luar 2017, judul kosong, atau isi kosong dikarantina
beserta alasannya. Sumber kosong dipulihkan dari domain URL; jika tidak tersedia,
diberi label `tidak diketahui`. Tanggal, judul, dan isi tidak diimputasi.


In [4]:
rejected = normalised.filter(pl.col('rejection_reason') != '')
df_missing_handled = normalised.filter(pl.col('rejection_reason') == '')
rejected_rows = int(rejected.select(pl.len()).collect(engine='streaming').item())
valid_rows = int(df_missing_handled.select(pl.len()).collect(engine='streaming').item())
sink_parquet_atomic(rejected, PROJECT_ROOT / 'data/processed/news_rejected.parquet')
rejection_counts = rejected.group_by('rejection_reason').len(name='artikel')
rejection_counts = rejection_counts.sort('artikel', descending=True).collect(engine='streaming')
rejection_counts.write_csv(PROJECT_ROOT / 'output/rejection_reasons.csv')
display(rejection_counts)
print(f'Valid: {valid_rows:,}; dikarantina: {rejected_rows:,}')
assert valid_rows > 0, 'Tidak ada artikel valid; periksa pemetaan kolom dan format tanggal.'
assert rejected_rows / raw_rows <= 0.05, (
    'Lebih dari 5% artikel ditolak. Periksa news_rejected.parquet dan format tanggal '
    'sebelum melanjutkan; batas ini tidak boleh dilonggarkan tanpa alasan.'
)


rejection_reason,artikel
str,u32
"""isi_kosong""",52


Valid: 455,626; dikarantina: 52


## 4. Deteksi dan Penanganan Outlier

Gunakan IQR jumlah kata pada artikel valid sebelum deduplikasi. Artikel panjang bisa
sah: simpan teks dan jumlah kata aslinya, lalu tambahkan flag `is_length_outlier`.


In [5]:
quantiles = df_missing_handled.select(
    pl.col('word_count').quantile(0.25, interpolation='linear').alias('q1'),
    pl.col('word_count').quantile(0.75, interpolation='linear').alias('q3'),
).collect(engine='streaming').row(0, named=True)
q1, q3 = quantiles['q1'], quantiles['q3']
iqr = q3 - q1
batas_bawah = max(0.0, q1 - 1.5 * iqr)
batas_atas = q3 + 1.5 * iqr
df_outlier_handled = df_missing_handled.with_columns(
    (~pl.col('word_count').is_between(batas_bawah, batas_atas)).alias('is_length_outlier')
)
print(f'Q1={q1:.2f}; Q3={q3:.2f}; IQR={iqr:.2f}')
print(f'Batas IQR: [{batas_bawah:.2f}, {batas_atas:.2f}] kata; outlier dipertahankan.')


Q1=188.00; Q3=321.00; IQR=133.00
Batas IQR: [0.00, 520.50] kata; outlier dipertahankan.


## 5. Deduplikasi

Kunci artikel: `(source, published_at, title_key, content)` setelah normalisasi.
Judul sama dengan teks berbeda tetap dipertahankan. Simpan baris pertama sesuai
urutan CSV dan arsipkan salinan yang dihapus melalui `row_id`.


In [6]:
df_dedup = df_outlier_handled.unique(subset=BUSINESS_KEY, keep='first', maintain_order=True)
sink_parquet_atomic(df_dedup.select(ANALYSIS_COLUMNS), CANDIDATE_PATH)
clean_candidate = pl.scan_parquet(CANDIDATE_PATH)
clean_rows = int(clean_candidate.select(pl.len()).collect(engine='streaming').item())
duplicate_rows = valid_rows - clean_rows
duplicates = df_outlier_handled.join(
    clean_candidate.select('row_id'), on='row_id', how='anti'
)
sink_parquet_atomic(duplicates, PROJECT_ROOT / 'data/processed/news_duplicates.parquet')
print(f'Valid sebelum dedup: {valid_rows:,}; salinan duplikat: {duplicate_rows:,}')
print(f'Artikel unik: {clean_rows:,}')


Valid sebelum dedup: 455,626; salinan duplikat: 266
Artikel unik: 455,360


## 6. Validasi Data Dasar

Validasi SQL harus menunjukkan nol pelanggaran. Rekonsiliasi jumlah baris:
mentah = dikarantina + duplikat valid + artikel bersih.


In [7]:
connection.execute(
    f'CREATE VIEW dataset_bersih AS SELECT * FROM read_parquet({sql_literal(CANDIDATE_PATH)})'
)
validasi = connection.execute("""
    SELECT
        COUNT(*) AS total_baris,
        COUNT(*) FILTER (WHERE published_at IS NULL OR YEAR(published_at) != 2017) AS tanggal_invalid,
        COUNT(*) FILTER (WHERE title IS NULL OR LENGTH(TRIM(title)) = 0) AS judul_kosong,
        COUNT(*) FILTER (WHERE content IS NULL OR LENGTH(TRIM(content)) = 0) AS isi_kosong,
        COUNT(*) FILTER (WHERE source IS NULL OR LENGTH(TRIM(source)) = 0) AS sumber_kosong,
        COUNT(*) FILTER (WHERE word_count IS NULL OR word_count <= 0 OR char_count < word_count) AS panjang_invalid,
        COUNT(*) FILTER (WHERE hour IS NOT NULL AND (hour < 0 OR hour > 23)) AS jam_invalid,
        COUNT(*) - COUNT(DISTINCT row_id) AS row_id_duplikat,
        COUNT(*) - COUNT(DISTINCT (source, published_at, title_key, content)) AS kunci_duplikat
    FROM dataset_bersih
""").pl()
validation_result = validasi.to_dicts()[0]
display(validasi)
assert validation_result['total_baris'] == clean_rows and clean_rows > 0
assert all(value == 0 for key, value in validation_result.items() if key != 'total_baris')
assert raw_rows == rejected_rows + duplicate_rows + clean_rows, 'Rekonsiliasi baris tidak sesuai.'
assert int(duplicates.select(pl.len()).collect(engine='streaming').item()) == duplicate_rows

profil_after = connection.execute('SUMMARIZE SELECT * FROM dataset_bersih').pl()
profil_after.write_csv(PROJECT_ROOT / 'output/profile_after_cleaning.csv')
display(profil_after)
clean_metrics = connection.execute("""
    SELECT MIN(published_date) AS date_min, MAX(published_date) AS date_max,
           COUNT(*) FILTER (WHERE is_length_outlier) AS length_outliers,
           COUNT(*) FILTER (WHERE source_is_imputed) AS source_unknown,
           COUNT(*) FILTER (WHERE NOT has_time) AS date_only_rows
    FROM dataset_bersih
""").pl().to_dicts()[0]
connection.close()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

total_baris,tanggal_invalid,judul_kosong,isi_kosong,sumber_kosong,panjang_invalid,jam_invalid,row_id_duplikat,kunci_duplikat
i64,i64,i64,i64,i64,i64,i64,i64,i64
455360,0,0,0,0,0,0,0,0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""row_id""","""UINTEGER""","""0""","""455677""",475106,"""227830.56013483836""","""131544.54815685196""","""115137""","""226345""","""340745""",455360,0.00
"""published_at""","""TIMESTAMP""","""2017-01-01 00:00:00""","""2017-12-31 23:58:25""",381447,"""2017-06-16 11:09:16.110183""",null,"""2017-03-21 20:33:48.102281""","""2017-06-10 03:52:53.713576""","""2017-08-31 01:59:41.626598""",455360,0.00
"""published_date""","""DATE""","""2017-01-01""","""2017-12-31""",380,"""2017-06-15 20:48:22.326072""",null,"""2017-03-21""","""2017-06-09""","""2017-08-31""",455360,0.00
"""month""","""DATE""","""2017-01-01""","""2017-12-01""",13,"""2017-06-01 05:58:50.365425""",null,"""2017-03-01""","""2017-06-01""","""2017-08-07""",455360,0.00
"""weekday""","""TINYINT""","""1""","""7""",7,"""3.595438773717498""","""1.8482899249739149""","""2""","""3""","""5""",455360,0.00
…,…,…,…,…,…,…,…,…,…,…,…
"""title_key""","""VARCHAR""",""""" di sini punya prinsip, jika …","""﻿﻿waspada akan diabetes mellit…",413082,null,null,null,null,null,455360,0.00
"""content""","""VARCHAR""",""""" CERITALAH ASEAN "" sedang mer…","""⁠Menteri Pekerjaan Umum dan Pe…",481886,null,null,null,null,null,455360,0.00
"""word_count""","""UINTEGER""","""1""","""10631""",1992,"""272.3464862965566""","""148.42720205549912""","""188""","""246""","""321""",455360,0.00


## 7. Simpan Data Bersih

Kandidat dipromosikan ke hasil akhir setelah validasi lulus. Metadata menyertakan
checksum input/output, keputusan cleaning, batas IQR, dan jumlah perubahan.


In [8]:
CANDIDATE_PATH.replace(OUTPUT_PATH)
cleaning_summary = {
    'source_raw_csv_sha256': PROFILE['raw_csv_sha256'],
    'source_raw_parquet_sha256': PROFILE['raw_parquet_sha256'],
    'clean_parquet_sha256': sha256_file(OUTPUT_PATH),
    'rows_raw': raw_rows, 'rows_rejected': rejected_rows,
    'rows_valid_before_dedup': valid_rows, 'duplicates_removed': duplicate_rows,
    'rows_clean': clean_rows, 'retained_percent': 100 * clean_rows / raw_rows,
    'business_key': BUSINESS_KEY, 'validation': validation_result,
    'rejection_counts': rejection_counts.to_dicts(),
    'iqr': {'q1': q1, 'q3': q3, 'lower': batas_bawah, 'upper': batas_atas,
            'population': 'valid articles before deduplication', 'action': 'flag and retain'},
    **clean_metrics,
}
write_json(PROJECT_ROOT / 'output/cleaning_summary.json', cleaning_summary)
print(f'Data bersih: {OUTPUT_PATH.relative_to(PROJECT_ROOT)}')


Data bersih: data/processed/dataset_bersih.parquet


## Catatan Milestone 2

Jumlah di bawah berasal dari proses cleaning yang baru dijalankan.
Flag outlier bukan bukti kesalahan data; baris karantina dan duplikat tersedia untuk audit.


In [9]:
display(Markdown(
    f"**{raw_rows:,} → {clean_rows:,} artikel**. Karantina: **{rejected_rows:,}**, "
    f"duplikat: **{duplicate_rows:,}**, outlier panjang yang dipertahankan: "
    f"**{clean_metrics['length_outliers']:,}**. "
    f"Sumber tidak diketahui: **{clean_metrics['source_unknown']:,}**. "
    "Semua aturan validasi SQL lulus dan jumlah baris sudah direkonsiliasi."
))


**455,678 → 455,360 artikel**. Karantina: **52**, duplikat: **266**, outlier panjang yang dipertahankan: **20,799**. Sumber tidak diketahui: **0**. Semua aturan validasi SQL lulus dan jumlah baris sudah direkonsiliasi.